# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset package using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and field `@id`s.

In [ ]:
# List all record sets, their @id, and the field @ids in the dataset
print("Available Record Sets and their fields:")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f'Record Set: {rs.id}')
    print(f'  name: {rs.name}')
    fields = [field.id for field in rs.fields]
    print(f'  Field @ids: {fields}')


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

In [ ]:
# Choose all record set @ids to extract
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

# Extract data from each record set
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded record set {rs_id} with shape: {dataframes[rs_id].shape}")
    except Exception as e:
        print(f"Error loading records for {rs_id}: {e}")

# For demonstration, pick the first non-empty record set
main_record_set_id = None
for rs_id in dataframes:
    df = dataframes[rs_id]
    if not df.empty:
        main_record_set_id = rs_id
        break

if main_record_set_id:
    print(f"First data columns in record set {main_record_set_id}: \n{dataframes[main_record_set_id].columns.tolist()}")
    dataframes[main_record_set_id].head()
else:
    print("No populated record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Pick a numeric field and a potential group field to demonstrate filtering and normalization.

In [ ]:
# Example: filter based on a numeric field, e.g., MW (molecular weight)
# Try to select a numeric field present in the main record set
import numpy as np

if main_record_set_id:
    df = dataframes[main_record_set_id].copy()
    numeric_fields = [col for col in df.columns if df[col].dtype in [np.float64, np.int64, float, int]]
    # If none detected as numeric use heuristic to cast certain columns
    if not numeric_fields:
        for c in df.columns:
            try:
                df[c] = pd.to_numeric(df[c])
                numeric_fields.append(c)
            except:
                continue

    print(f"Numeric fields detected: {numeric_fields}")
    if numeric_fields:
        numeric_field = numeric_fields[0]
        # Remove NaNs
        df = df.dropna(subset=[numeric_field])
        threshold = df[numeric_field].mean() if df[numeric_field].mean() > 0 else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records in {main_record_set_id} with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()

        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by a categorical/string field if available
        string_fields = [col for col in filtered_df.columns if filtered_df[col].dtype == object]
        group_field = None
        for col in string_fields:
            if filtered_df[col].nunique() > 1 and filtered_df[col].nunique() < 30:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable categorical field to group by.")
    else:
        print("No numeric fields found for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In this example, we plot the distribution of the main numeric field and its relationship with another variable (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_fields:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field], bins=30)
    plt.title(f"Distribution of {numeric_field} in {main_record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Optionally show relationship between two numeric fields
    if len(numeric_fields) > 1:
        plt.figure(figsize=(6, 6))
        sns.scatterplot(
            x=df[numeric_fields[0]], y=df[numeric_fields[1]]
        )
        plt.xlabel(numeric_fields[0])
        plt.ylabel(numeric_fields[1])
        plt.title(f"{numeric_fields[0]} vs {numeric_fields[1]}")
        plt.show()
else:
    print("Not enough fields for visualization.")

## 6. Conclusion
This notebook demonstrated end-to-end metadata exploration, record retrieval, and basic analysis using the `mlcroissant` library, referencing record sets and fields by their `@id` as per Croissant best practices.

Key findings:
- The dataset includes protein-level mass spectrometry data related to human mast cell extracellular vesicles.
- We loaded record sets, explored available field `@id`s, performed filtering and normalization on a chosen numeric field, and visualized its distribution.

Further analysis can be performed based on domain-specific questions and available data fields.